# Rebuttal supporting tables, including latest HANNA

This notebook reproduces the addtional supporting analyses performed in the rebuttal and included in the paper.


In [1]:
# Run from the repository root or notebooks_cr; every write stays in notebooks_cr.
import json
import sys
from pathlib import Path
_PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file())
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
from notebooks.analysis_support import (
    RUNS, PROJECT_ROOT, open_results, output_dir, leaderboard as read_leaderboard,
    pairwise_table,
)
ARCHIVES = open_results()
NOTEBOOK_OUTPUT = output_dir("rebuttal_supporting_tables")
# Completion records and effective settings come from the public result reader.
provenance = {dataset: {
    **result.machine_summary(),
    "effective_config": result.resolved_config,
} for dataset, result in ARCHIVES.items()}
(NOTEBOOK_OUTPUT / "input_provenance.json").write_text(
    json.dumps(provenance, indent=2), encoding="utf-8"
)
print("Outputs:", NOTEBOOK_OUTPUT)


Outputs: /workspaces/TheRulerAndTheJudge-main/notebooks/outputs/rebuttal_supporting_tables


In [2]:
from __future__ import annotations

from pathlib import Path
from itertools import combinations, product
from typing import Dict, Iterable, Optional, Tuple
import json

import numpy as np
import pandas as pd
from IPython.display import display

# =============================================================================
# 0. CONFIGURATION
# =============================================================================
ROOT = NOTEBOOK_OUTPUT
EXPERIMENTS = {dataset: RUNS[dataset] for dataset in ("HANNA", "HiTZ-BASSE", "SummEval")}
OUT = NOTEBOOK_OUTPUT
OUT.mkdir(parents=True, exist_ok=True)

# Baseline operational choices used in the paper/rebuttal.
BASE_TAU_H = 0.80
BASE_TAU_L = 0.80
BASE_FSR_MAX = 0.30
BASE_MNSQ = (0.50, 1.50)

# Reviewer-3 operational sensitivity grid.
TAU_H_VALUES = [0.70, 0.75, 0.80, 0.85, 0.90]
TAU_L_VALUES = [0.70, 0.75, 0.80, 0.85, 0.90]
FSR_MAX_VALUES = [0.20, 0.25, 0.30, 0.35, 0.40]
MNSQ_INTERVALS = [(0.50, 1.30), (0.50, 1.50), (0.50, 2.00)]

# =============================================================================
# 1. UTILITIES
# =============================================================================
def _norm_text(x) -> str:
    return str(x).strip()


def _key(x) -> str:
    return _norm_text(x).casefold()


def _first_existing(df: pd.DataFrame, names: Iterable[str], *, required: bool = True) -> Optional[str]:
    by_key = {_key(c): c for c in df.columns}
    for name in names:
        if _key(name) in by_key:
            return by_key[_key(name)]
    if required:
        raise KeyError(f'None of {list(names)} found. Available columns: {list(df.columns)}')
    return None


def _numeric(df: pd.DataFrame, col: str) -> pd.Series:
    return pd.to_numeric(df[col], errors='coerce')


def _safe_div(num: float, den: float) -> float:
    return float(num / den) if den else np.nan


def _json_list(values: Iterable[str]) -> str:
    return json.dumps([str(v) for v in values], ensure_ascii=False)


def save_table(df: pd.DataFrame, stem: str, *, latex: bool = False) -> None:
    """Save both CSV and Parquet; optionally a compact LaTeX version."""
    csv_path = OUT / f'{stem}.csv'
    parquet_path = OUT / f'{stem}.parquet'
    df.to_csv(csv_path, index=False)
    df.to_parquet(parquet_path, index=False)
    if latex:
        (OUT / f'{stem}.tex').write_text(
            df.to_latex(index=False, escape=True, float_format=lambda x: f'{x:.3f}'),
            encoding='utf-8',
        )


# =============================================================================
# 2. LOAD AND CANONICALIZE THE THREE RESULT FOLDERS
# =============================================================================
def load_analysis_tables(dataset: str, folder: Path) -> Dict[str, pd.DataFrame]:
    required = [
        'benchmark_cards.parquet',
        'llm_audit.parquet',
        'leaderboard_results.parquet',
        'loocv_posterior_artifacts_manifest.parquet',
    ]
    missing = [name for name in required if not (folder / name).exists()]
    if missing:
        raise FileNotFoundError(f'{dataset}: missing {missing} in {folder}')

    cards_raw = pd.read_parquet(folder / 'benchmark_cards.parquet')
    audit_raw = pd.read_parquet(folder / 'llm_audit.parquet')
    lb_raw = pd.read_parquet(folder / 'leaderboard_results.parquet')
    manifest = pd.read_parquet(folder / 'loocv_posterior_artifacts_manifest.parquet')

    # ---- Stage 1 -------------------------------------------------------------
    cards = pd.DataFrame({
        'dataset': dataset,
        'dimension': cards_raw[_first_existing(cards_raw, ['dimension', 'Dimension'])].astype(str),
        'resolution_label': cards_raw[_first_existing(cards_raw, ['resolution_label'])].astype(str),
        'spread80': _numeric(cards_raw, _first_existing(cards_raw, ['spread80'])),
        'csem80': _numeric(cards_raw, _first_existing(cards_raw, ['csem80'])),
        'delta95': _numeric(cards_raw, _first_existing(cards_raw, ['delta95'])),
        'resolution_ratio': _numeric(cards_raw, _first_existing(cards_raw, ['resolution_ratio'])),
        'theta_reliability': _numeric(cards_raw, _first_existing(cards_raw, ['theta_reliability'])),
        'stage1_supported_resolution': _numeric(cards_raw, _first_existing(cards_raw, ['supported_resolution'])),
        'tau_h_stage1': _numeric(cards_raw, _first_existing(cards_raw, ['supported_resolution_tau_h'])),
        'stage1_n_supported_pairs': _numeric(cards_raw, _first_existing(cards_raw, ['n_supported_pairs'])),
        'stage1_n_pairs': _numeric(cards_raw, _first_existing(cards_raw, ['n_pairs'])),
    })
    cards['_dimension_key'] = cards['dimension'].map(_key)

    # ---- Stage 2 -------------------------------------------------------------
    audit = pd.DataFrame({
        'dataset': dataset,
        'dimension': audit_raw[_first_existing(audit_raw, ['dimension', 'Dimension'])].astype(str),
        'judge_model': audit_raw[_first_existing(audit_raw, ['judge_model', 'Model', 'rater_id'])].astype(str),
        'infit_mnsq': _numeric(audit_raw, _first_existing(audit_raw, ['Infit_MNSQ', 'infit_mnsq', 'Infit MNSQ'])),
        'outfit_mnsq': _numeric(audit_raw, _first_existing(audit_raw, ['Outfit_MNSQ', 'outfit_mnsq', 'Outfit MNSQ'])),
    })
    n_col = _first_existing(audit_raw, ['N', 'n'], required=False)
    audit['n_stage2_observations'] = _numeric(audit_raw, n_col) if n_col else np.nan
    audit['_dimension_key'] = audit['dimension'].map(_key)
    audit['_model_key'] = audit['judge_model'].map(_key)
    # Exactly one Stage-2 audit row must exist per dataset/dimension/judge.
    audit_keys = ['dataset', '_dimension_key', '_model_key']

    audit_duplicates = audit[
        audit.duplicated(audit_keys, keep=False)
    ].sort_values(audit_keys)

    if not audit_duplicates.empty:
        duplicate_keys = (
            audit_duplicates[audit_keys]
            .drop_duplicates()
            .to_dict('records')
        )
        raise ValueError(
            f"{dataset}: duplicate Stage-2 rows found for "
            f"{duplicate_keys}"
        )

    # ---- Stage 3 -------------------------------------------------------------
    dim_col = _first_existing(lb_raw, ['Dimension', 'dimension'])
    model_col = _first_existing(lb_raw, ['Model', 'judge_model', 'model'])

    canonical_candidates = {
        'nsmf': ['nSMF', 'NSMF', 'NFSM', 'nfsm'],
        'spearman': ['Spearman', 'spearman'],
        'raw_spearman': ['Raw-score Spearman', 'Raw score Spearman', 'raw_score_spearman'],
        'fsr_stored': ['False Separation Rate', 'Support FSR', 'FSR', 'fsr'],
        'emd': ['EMD', 'emd'],
        'pairacc_theta': ['PairAcc theta', 'pairacc_theta'],
        'pairacc_tau_stored': ['PairAcc tau', 'pairacc_tau'],
        'human_sr_stored': ['Human SR', 'human_sr'],
        'judge_sr_stored': ['Judge SR', 'judge_sr'],
        'support_precision_stored': ['Support precision', 'support_precision'],
        'support_recall_stored': ['Support recall', 'support_recall'],
        'sda_stored': ['Supported Direction Accuracy', 'supported_direction_accuracy'],
    }

    lb = pd.DataFrame({
        'dataset': dataset,
        'dimension': lb_raw[dim_col].astype(str),
        'judge_model': lb_raw[model_col].astype(str),
    })
    for new_col, candidates in canonical_candidates.items():
        old_col = _first_existing(lb_raw, candidates, required=False)
        lb[new_col] = _numeric(lb_raw, old_col) if old_col else np.nan

    nsys_col = _first_existing(lb_raw, ['N systems', 'n_systems'], required=False)
    lb['n_systems'] = _numeric(lb_raw, nsys_col) if nsys_col else np.nan
    lb['_dimension_key'] = lb['dimension'].map(_key)
    lb['_model_key'] = lb['judge_model'].map(_key)

    # Exactly one Stage-3 row must exist per dataset/dimension/judge.
    lb_keys = ['dataset', '_dimension_key', '_model_key']

    lb_duplicates = lb[
        lb.duplicated(lb_keys, keep=False)
    ].sort_values(lb_keys)

    if not lb_duplicates.empty:
        duplicate_keys = (
            lb_duplicates[lb_keys]
            .drop_duplicates()
            .to_dict('records')
        )
        raise ValueError(
            f"{dataset}: duplicate leaderboard rows found for "
            f"{duplicate_keys}"
        )

    manifest = manifest.copy()
    manifest['dataset'] = dataset
    manifest['_dimension_key'] = manifest['dimension'].map(_key)
    manifest['_model_key'] = manifest['judge_model'].map(_key)

    return {
        'folder': folder,
        'cards': cards,
        'audit': audit,
        'leaderboard': lb,
        'manifest': manifest,
    }


RESULTS = {dataset: load_analysis_tables(dataset, folder) for dataset, folder in EXPERIMENTS.items()}
CARDS = pd.concat([x['cards'] for x in RESULTS.values()], ignore_index=True)
AUDIT = pd.concat([x['audit'] for x in RESULTS.values()], ignore_index=True)
LEADERBOARD = pd.concat([x['leaderboard'] for x in RESULTS.values()], ignore_index=True)

print('Loaded datasets:', ', '.join(RESULTS))
print('Stage-1 rows:', len(CARDS), '| Stage-2 rows:', len(AUDIT), '| Stage-3 rows:', len(LEADERBOARD))


# =============================================================================
# 3. RECOMPUTE PAIRWISE MID-DOMINANCE FROM COMPACT POSTERIORS
#    This opens the NPZ lazily and reads only post_theta/post_human vectors.
# =============================================================================
def _normalized_posterior(arr) -> np.ndarray:
    p = np.asarray(arr, dtype=np.float64).reshape(-1)
    if not np.all(np.isfinite(p)) or np.any(p < 0):
        raise ValueError('Invalid posterior mass vector.')
    total = p.sum()
    if total <= 0:
        raise ValueError('Posterior has zero total mass.')
    return p / total


def build_pairwise_table(dataset, result):
    return pairwise_table(dataset, result['folder'])


PAIRWISE = pd.concat(
    [build_pairwise_table(dataset, result) for dataset, result in RESULTS.items()],
    ignore_index=True,
)
save_table(PAIRWISE, 'pairwise_mid_dominance_all_datasets')
print('Pairwise rows:', len(PAIRWISE))


# =============================================================================
# 4. SUPPORT METRICS AT ARBITRARY tau_H / tau_L
# =============================================================================
def support_metrics_for_thresholds(
    pairwise: pd.DataFrame,
    tau_h: float,
    tau_l: float,
) -> pd.DataFrame:
    rows = []
    group_cols = ['dataset', '_dimension_key', '_model_key']

    for keys, g in pairwise.groupby(group_cols, sort=True):
        dataset, dim_key, model_key = keys
        mh = g['human_signed_margin'].to_numpy(float)
        mj = g['judge_signed_margin'].to_numpy(float)

        h_sup = np.abs(mh) >= tau_h
        j_sup = np.abs(mj) >= tau_l
        same_direction = np.sign(mh) == np.sign(mj)

        tp = int(np.sum(h_sup & j_sup))
        fp = int(np.sum((~h_sup) & j_sup))
        fn = int(np.sum(h_sup & (~j_sup)))
        tn = int(np.sum((~h_sup) & (~j_sup)))
        n = len(g)

        supported_both = h_sup & j_sup
        ternary_h = np.where(h_sup, np.sign(mh), 0)
        ternary_j = np.where(j_sup, np.sign(mj), 0)

        rows.append({
            'dataset': dataset,
            'dimension': str(g['dimension'].iloc[0]),
            'judge_model': str(g['judge_model'].iloc[0]),
            '_dimension_key': dim_key,
            '_model_key': model_key,
            'tau_h': float(tau_h),
            'tau_l': float(tau_l),
            'n_pairs': int(n),
            'support_tp': tp,
            'support_fp': fp,
            'support_fn': fn,
            'support_tn': tn,
            'human_sr': _safe_div(tp + fn, n),
            'judge_sr': _safe_div(tp + fp, n),
            'support_precision': _safe_div(tp, tp + fp),
            'support_recall': _safe_div(tp, tp + fn),
            'fsr': _safe_div(fp, fp + tn),
            'supported_direction_accuracy': _safe_div(
                int(np.sum(supported_both & same_direction)), int(np.sum(supported_both))
            ),
            'pairacc_tau': float(np.mean(ternary_h == ternary_j)),
        })

    return pd.DataFrame(rows)


SUPPORT_CACHE = {
    (tau_h, tau_l): support_metrics_for_thresholds(PAIRWISE, tau_h, tau_l)
    for tau_h, tau_l in product(TAU_H_VALUES, TAU_L_VALUES)
}
BASE_SUPPORT = SUPPORT_CACHE[(BASE_TAU_H, BASE_TAU_L)]
save_table(BASE_SUPPORT, 'corrected_support_metrics_tau_0p8')

LB_CORRECTED = LEADERBOARD.merge(
    BASE_SUPPORT[
        [
            'dataset', '_dimension_key', '_model_key', 'fsr', 'human_sr', 'judge_sr',
            'support_precision', 'support_recall', 'supported_direction_accuracy',
            'pairacc_tau', 'support_tp', 'support_fp', 'support_fn', 'support_tn', 'n_pairs',
        ]
    ],
    on=['dataset', '_dimension_key', '_model_key'],
    how='left',
    validate='one_to_one',
)


# =============================================================================
# 5. OPERATIONAL SELECTION RULES
# =============================================================================
def stage2_pass_table(low: float, high: float) -> pd.DataFrame:
    out = AUDIT.copy()
    out['stage2_pass'] = (
        out['infit_mnsq'].between(low, high, inclusive='both')
        & out['outfit_mnsq'].between(low, high, inclusive='both')
    )
    out['mnsq_low'] = float(low)
    out['mnsq_high'] = float(high)
    return out


def _pick_max(df: pd.DataFrame, metric: str) -> Tuple[Optional[str], float, int]:
    valid = df[['judge_model', metric]].dropna().copy()
    if valid.empty:
        return None, np.nan, 0
    best = valid[metric].max()
    tied = valid[np.isclose(valid[metric], best, rtol=0.0, atol=1e-12)].sort_values('judge_model')
    return str(tied.iloc[0]['judge_model']), float(best), int(len(tied))


def _stage1_resolution_status(label: str) -> str:
    """
    Map benchmark-card resolution labels to operational Stage-1 states.
    """
    normalized = (
        _key(label)
        .replace('_', ' ')
        .replace('-', ' ')
    )

    if 'unacceptable' in normalized:
        return 'unacceptable'

    if 'fine' in normalized and 'grain' in normalized:
        return 'fine-grained'

    return 'warning'


def build_decision_table(
    support_table: pd.DataFrame,
    fsr_max: float,
    mnsq_interval: Tuple[float, float],
) -> pd.DataFrame:
    low, high = mnsq_interval

    gate = stage2_pass_table(low, high)[
        [
            'dataset',
            '_dimension_key',
            '_model_key',
            'stage2_pass',
            'infit_mnsq',
            'outfit_mnsq',
        ]
    ]

    work = (
        LEADERBOARD
        .drop(columns=['fsr_stored'], errors='ignore')
        .merge(
            support_table[
                [
                    'dataset',
                    '_dimension_key',
                    '_model_key',
                    'fsr',
                    'human_sr',
                    'judge_sr',
                ]
            ],
            on=[
                'dataset',
                '_dimension_key',
                '_model_key',
            ],
            how='left',
            validate='one_to_one',
        )
        .merge(
            gate,
            on=[
                'dataset',
                '_dimension_key',
                '_model_key',
            ],
            how='left',
            validate='one_to_one',
        )
    )

    work['stage2_pass'] = (
        work['stage2_pass']
        .fillna(False)
        .astype(bool)
    )

    rows = []

    for (dataset, dim_key), g in work.groupby(
        ['dataset', '_dimension_key'],
        sort=True,
    ):
        dimension = str(g['dimension'].iloc[0])

        # Stage 2 is a hard admissibility gate.
        admitted = g[g['stage2_pass']].copy()

        # Stage 3 risk selection additionally requires FSR <= gamma.
        risk_eligible = admitted[
            admitted['fsr'] <= fsr_max
        ].copy()

        # Conventional comparison: raw-score correlation.
        raw_model, raw_value, raw_ties = _pick_max(
            g,
            'raw_spearman',
        )

        # Latent Spearman before applying Stage 2.
        latent_all_model, latent_all_value, latent_all_ties = (
            _pick_max(
                g,
                'spearman',
            )
        )

        # Conditional Stage-3 winners after Stage-2 admissibility.
        conditional_rank_model, conditional_rank_value, rank_ties = (
            _pick_max(
                admitted,
                'spearman',
            )
        )

        (
            conditional_pairwise_model,
            conditional_pairwise_value,
            pairwise_ties,
        ) = _pick_max(
            admitted,
            'nsmf',
        )

        (
            conditional_risk_model,
            conditional_risk_nsmf,
            risk_ties,
        ) = _pick_max(
            risk_eligible,
            'nsmf',
        )

        conditional_risk_fsr = np.nan

        if conditional_risk_model is not None:
            conditional_risk_fsr = float(
                risk_eligible.loc[
                    risk_eligible['judge_model']
                    == conditional_risk_model,
                    'fsr',
                ].iloc[0]
            )

        card = CARDS[
            (CARDS['dataset'] == dataset)
            & (CARDS['_dimension_key'] == dim_key)
        ]

        resolution_label = (
            str(card['resolution_label'].iloc[0])
            if len(card)
            else 'unknown'
        )

        stage1_status = _stage1_resolution_status(
            resolution_label
        )

        # -------------------------------------------------------------
        # Operational decision
        # -------------------------------------------------------------
        # Conditional winners are retained above for diagnostic tables.
        # Operational selections are suppressed when Stage 1 refuses.
        if stage1_status == 'unacceptable':
            rank_model = None
            rank_value = np.nan

            pairwise_model = None
            pairwise_value = np.nan

            risk_model = None
            risk_nsmf = np.nan
            risk_fsr = np.nan

            outcome = (
                'refuse: Stage 1 ruler unacceptable'
            )

        else:
            rank_model = conditional_rank_model
            rank_value = conditional_rank_value

            pairwise_model = conditional_pairwise_model
            pairwise_value = conditional_pairwise_value

            risk_model = conditional_risk_model
            risk_nsmf = conditional_risk_nsmf
            risk_fsr = conditional_risk_fsr

            if admitted.empty:
                outcome = (
                    'refuse: no Stage-2-admissible judge'
                )

            elif risk_model is None:
                outcome = (
                    'abstain: no Stage-2-admissible '
                    'judge satisfies FSR bound'
                )

            elif stage1_status == 'warning':
                outcome = (
                    f'warn/select: Stage 1 '
                    f'{resolution_label}'
                )

            else:
                outcome = 'select'

        framework_refused = outcome.startswith(
            'refuse:'
        )

        risk_abstained = outcome.startswith(
            'abstain:'
        )

        rows.append(
            {
                'dataset': dataset,
                'dimension': dimension,
                '_dimension_key': dim_key,

                'stage1_resolution_label': resolution_label,
                'stage1_status': stage1_status,

                'mnsq_low': float(low),
                'mnsq_high': float(high),
                'fsr_max': float(fsr_max),

                'n_judges': int(len(g)),
                'n_stage2_pass': int(
                    admitted['judge_model'].nunique()
                ),

                # ---------------------------------------------
                # Conventional/raw correlation
                # ---------------------------------------------
                'raw_spearman_selected_model': raw_model,
                'raw_spearman_selected_value': raw_value,
                'raw_spearman_n_tied_best': raw_ties,

                # ---------------------------------------------
                # Latent Spearman before Stage-2 gating
                # ---------------------------------------------
                'latent_spearman_all_selected_model':
                    latent_all_model,

                'latent_spearman_all_selected_value':
                    latent_all_value,

                'latent_spearman_all_n_tied_best':
                    latent_all_ties,

                # ---------------------------------------------
                # Diagnostic conditional Stage-3 winners
                # ---------------------------------------------
                'conditional_rank_selected_model':
                    conditional_rank_model,

                'conditional_rank_selected_spearman':
                    conditional_rank_value,

                'conditional_pairwise_selected_model':
                    conditional_pairwise_model,

                'conditional_pairwise_selected_nsmf':
                    conditional_pairwise_value,

                'conditional_risk_selected_model':
                    conditional_risk_model,

                'conditional_risk_selected_nsmf':
                    conditional_risk_nsmf,

                'conditional_risk_selected_fsr':
                    conditional_risk_fsr,

                # ---------------------------------------------
                # Operational framework outputs
                # ---------------------------------------------
                'rank_selected_model': rank_model,
                'rank_selected_spearman': rank_value,
                'rank_n_tied_best': rank_ties if rank_model is not None else 0,
                
                'pairwise_selected_model': pairwise_model,
                'pairwise_selected_nsmf': pairwise_value,
                'pairwise_n_tied_best': pairwise_ties if pairwise_model is not None else 0,

                'risk_selected_model': risk_model,
                'risk_selected_nsmf': risk_nsmf,
                'risk_selected_fsr': risk_fsr,
                'risk_n_tied_best': risk_ties if risk_model is not None else 0,

                'decision_outcome': outcome,
                'framework_refused': framework_refused,
                'risk_abstained': risk_abstained,
                'risk_no_selection': risk_model is None,

                # ---------------------------------------------
                # Reviewer-1 comparison indicators
                # ---------------------------------------------
                'raw_vs_latent_changed':
                    raw_model != latent_all_model,

                'raw_vs_framework_rank_changed':
                    raw_model != rank_model,

                'latent_vs_framework_rank_changed':
                    latent_all_model != rank_model,

                'raw_vs_pairwise_changed':
                    raw_model != pairwise_model,

                'raw_vs_risk_changed_or_no_selection':
                    raw_model != risk_model,
            }
        )

    return pd.DataFrame(rows)


BASE_DECISIONS = build_decision_table(
    BASE_SUPPORT,
    fsr_max=BASE_FSR_MAX,
    mnsq_interval=BASE_MNSQ,
)
save_table(BASE_DECISIONS.drop(columns=['_dimension_key']), 'all_datasets_baseline_decisions', latex=True)


# =============================================================================
# 6. HANNA SUPPORTING TABLES (Reviewers 1, 2 and 3)
# =============================================================================
HANNA_STAGE1 = (
    CARDS[CARDS['dataset'] == 'HANNA']
    [[
        'dimension', 'resolution_label', 'spread80', 'csem80', 'delta95',
        'resolution_ratio', 'theta_reliability', 'stage1_supported_resolution',
        'stage1_n_supported_pairs', 'stage1_n_pairs',
    ]]
    .sort_values('dimension')
    .reset_index(drop=True)
)
HANNA_STAGE1['supported_pairs'] = (
    HANNA_STAGE1['stage1_n_supported_pairs'].astype('Int64').astype(str)
    + '/'
    + HANNA_STAGE1['stage1_n_pairs'].astype('Int64').astype(str)
)
save_table(HANNA_STAGE1, 'hanna_stage1_benchmark_resolution', latex=True)

HANNA_STAGE2 = (
    stage2_pass_table(*BASE_MNSQ)
    .query("dataset == 'HANNA'")
    [[
        'dimension', 'judge_model', 'infit_mnsq', 'outfit_mnsq',
        'stage2_pass', 'n_stage2_observations',
    ]]
    .sort_values(['dimension', 'stage2_pass', 'infit_mnsq'], ascending=[True, False, True])
    .reset_index(drop=True)
)
save_table(HANNA_STAGE2, 'hanna_stage2_judge_fit_all', latex=False)

HANNA_STAGE2_SUMMARY = (
    HANNA_STAGE2.groupby('dimension', as_index=False)
    .agg(
        n_judges=('judge_model', 'nunique'),
        n_stage2_pass=('stage2_pass', 'sum'),
        passing_judges=('judge_model', lambda s: _json_list(
            HANNA_STAGE2.loc[s.index].loc[
                HANNA_STAGE2.loc[s.index, 'stage2_pass'], 'judge_model'
            ]
        )),
        min_infit=('infit_mnsq', 'min'),
        max_infit=('infit_mnsq', 'max'),
        min_outfit=('outfit_mnsq', 'min'),
        max_outfit=('outfit_mnsq', 'max'),
    )
)
save_table(HANNA_STAGE2_SUMMARY, 'hanna_stage2_summary', latex=True)

HANNA_STAGE3 = (
    LB_CORRECTED.query("dataset == 'HANNA'")
    .merge(
        stage2_pass_table(*BASE_MNSQ)[
            ['dataset', '_dimension_key', '_model_key', 'stage2_pass', 'infit_mnsq', 'outfit_mnsq']
        ],
        on=['dataset', '_dimension_key', '_model_key'],
        how='left',
        validate='one_to_one',
    )
    [[
        'dimension', 'judge_model', 'stage2_pass', 'infit_mnsq', 'outfit_mnsq',
        'raw_spearman', 'spearman', 'nsmf', 'fsr', 'human_sr', 'judge_sr',
        'support_precision', 'support_recall', 'supported_direction_accuracy',
        'pairacc_tau', 'pairacc_theta', 'emd',
    ]]
    .sort_values(['dimension', 'nsmf'], ascending=[True, False])
    .reset_index(drop=True)
)
save_table(HANNA_STAGE3, 'hanna_stage3_all_judges', latex=False)

HANNA_DECISIONS = (
    BASE_DECISIONS.query("dataset == 'HANNA'")
    .drop(columns=['dataset', '_dimension_key'])
    .sort_values('dimension')
    .reset_index(drop=True)
)
save_table(HANNA_DECISIONS, 'hanna_operational_decisions', latex=True)


def top_k_long(df: pd.DataFrame, metrics: Dict[str, bool], k: int = 3) -> pd.DataFrame:
    """metrics maps column -> ascending (False means larger is better)."""
    rows = []
    for dimension, gd in df.groupby('dimension', sort=True):
        for metric, ascending in metrics.items():
            valid = gd.dropna(subset=[metric]).sort_values(
                [metric, 'judge_model'], ascending=[ascending, True]
            ).head(k)
            for rank, rec in enumerate(valid.itertuples(index=False), start=1):
                rows.append({
                    'dimension': dimension,
                    'metric': metric,
                    'rank': rank,
                    'judge_model': rec.judge_model,
                    'value': float(getattr(rec, metric)),
                    'stage2_pass': bool(rec.stage2_pass),
                })
    return pd.DataFrame(rows)


HANNA_TOP3 = top_k_long(
    HANNA_STAGE3,
    metrics={
        'raw_spearman': False,
        'spearman': False,
        'nsmf': False,
        'fsr': True,
        'emd': True,
    },
    k=3,
)
save_table(HANNA_TOP3, 'hanna_stage3_top3_by_metric', latex=False)


# =============================================================================
# 7. REVIEWER-1: HOW OFTEN DOES THE FRAMEWORK CHANGE THE DECISION?
# =============================================================================
RANKING_DECISION_COMPARISON = BASE_DECISIONS[
    [
        'dataset',
        'dimension',
        'stage1_resolution_label',
        'stage1_status',
        'n_stage2_pass',

        'raw_spearman_selected_model',
        'raw_spearman_selected_value',

        'latent_spearman_all_selected_model',
        'latent_spearman_all_selected_value',

        # Diagnostic conditional winners
        'conditional_rank_selected_model',
        'conditional_rank_selected_spearman',

        'conditional_pairwise_selected_model',
        'conditional_pairwise_selected_nsmf',

        'conditional_risk_selected_model',
        'conditional_risk_selected_nsmf',
        'conditional_risk_selected_fsr',

        # Operational decisions
        'rank_selected_model',
        'rank_selected_spearman',

        'pairwise_selected_model',
        'pairwise_selected_nsmf',

        'risk_selected_model',
        'risk_selected_nsmf',
        'risk_selected_fsr',

        'decision_outcome',
        'framework_refused',
        'risk_abstained',

        'raw_vs_latent_changed',
        'raw_vs_framework_rank_changed',
        'latent_vs_framework_rank_changed',
        'raw_vs_pairwise_changed',
        'raw_vs_risk_changed_or_no_selection',
    ]
].sort_values(
    ['dataset', 'dimension']
).reset_index(drop=True)

save_table(
    RANKING_DECISION_COMPARISON,
    'reviewer1_decision_change_by_dataset_dimension',
    latex=True,
)

_risk_sel = BASE_DECISIONS['risk_selected_model'].notna()
_risk_diff = (
    BASE_DECISIONS.loc[_risk_sel, 'raw_spearman_selected_model']
    != BASE_DECISIONS.loc[_risk_sel, 'risk_selected_model']
)

DECISION_CHANGE_SUMMARY = pd.DataFrame(
    [
        {
            'comparison': (
                'Raw-score Spearman winner vs '
                'latent Spearman winner (all judges)'
            ),
            'changed': int(
                BASE_DECISIONS[
                    'raw_vs_latent_changed'
                ].sum()
            ),
            'total': int(len(BASE_DECISIONS)),
            'fraction_changed': float(
                BASE_DECISIONS[
                    'raw_vs_latent_changed'
                ].mean()
            ),
        },
        {
            'comparison': (
                'Raw-score Spearman winner vs '
                'risk-sensitive winner '
                '(conditional on a judge being selected)'
            ),
            'changed': int(_risk_diff.sum()),
            'total': int(_risk_sel.sum()),
            'fraction_changed': float(_risk_diff.mean()),
        },
        {
            'comparison': (
                'Raw-score Spearman winner vs '
                'operational framework rank outcome'
            ),
            'changed': int(
                BASE_DECISIONS[
                    'raw_vs_framework_rank_changed'
                ].sum()
            ),
            'total': int(len(BASE_DECISIONS)),
            'fraction_changed': float(
                BASE_DECISIONS[
                    'raw_vs_framework_rank_changed'
                ].mean()
            ),
        },
        {
            'comparison': (
                'Latent Spearman winner vs '
                'operational framework rank outcome'
            ),
            'changed': int(
                BASE_DECISIONS[
                    'latent_vs_framework_rank_changed'
                ].sum()
            ),
            'total': int(len(BASE_DECISIONS)),
            'fraction_changed': float(
                BASE_DECISIONS[
                    'latent_vs_framework_rank_changed'
                ].mean()
            ),
        },
        {
            'comparison': (
                'Raw-score Spearman winner vs '
                'operational pairwise-fidelity outcome'
            ),
            'changed': int(
                BASE_DECISIONS[
                    'raw_vs_pairwise_changed'
                ].sum()
            ),
            'total': int(len(BASE_DECISIONS)),
            'fraction_changed': float(
                BASE_DECISIONS[
                    'raw_vs_pairwise_changed'
                ].mean()
            ),
        },
        {
            'comparison': (
                'Raw-score Spearman winner vs '
                'risk-sensitive outcome '
                '(different/refuse/abstain)'
            ),
            'changed': int(
                BASE_DECISIONS[
                    'raw_vs_risk_changed_or_no_selection'
                ].sum()
            ),
            'total': int(len(BASE_DECISIONS)),
            'fraction_changed': float(
                BASE_DECISIONS[
                    'raw_vs_risk_changed_or_no_selection'
                ].mean()
            ),
        },
        {
            'comparison': 'Framework refusals',
            'changed': int(
                BASE_DECISIONS[
                    'framework_refused'
                ].sum()
            ),
            'total': int(len(BASE_DECISIONS)),
            'fraction_changed': float(
                BASE_DECISIONS[
                    'framework_refused'
                ].mean()
            ),
        },
        {
            'comparison': (
                'Risk-sensitive abstentions after '
                'Stages 1 and 2 pass'
            ),
            'changed': int(
                BASE_DECISIONS[
                    'risk_abstained'
                ].sum()
            ),
            'total': int(len(BASE_DECISIONS)),
            'fraction_changed': float(
                BASE_DECISIONS[
                    'risk_abstained'
                ].mean()
            ),
        },
    ]
)

save_table(
    DECISION_CHANGE_SUMMARY,
    'reviewer1_decision_change_summary',
    latex=True,
)


# =============================================================================
# 8. REVIEWER-3: OPERATIONAL SENSITIVITY ANALYSIS
# =============================================================================
# 8a. Full factorial grid (375 configurations, no refitting).
full_grid_parts = []
for tau_h, tau_l, fsr_max, mnsq_interval in product(
    TAU_H_VALUES, TAU_L_VALUES, FSR_MAX_VALUES, MNSQ_INTERVALS
):
    d = build_decision_table(
        SUPPORT_CACHE[(tau_h, tau_l)],
        fsr_max=fsr_max,
        mnsq_interval=mnsq_interval,
    )
    d['tau_h'] = tau_h
    d['tau_l'] = tau_l
    d['config_id'] = (
        f'th={tau_h:.2f}|tl={tau_l:.2f}|fsr={fsr_max:.2f}|'
        f'mnsq={mnsq_interval[0]:.2f}-{mnsq_interval[1]:.2f}'
    )
    full_grid_parts.append(d)

SENSITIVITY_FULL_GRID = pd.concat(full_grid_parts, ignore_index=True)
save_table(SENSITIVITY_FULL_GRID.drop(columns=['_dimension_key']), 'reviewer3_sensitivity_full_grid')

# Compare every configuration to the corrected baseline.
baseline_for_merge = BASE_DECISIONS[
    [
        'dataset', '_dimension_key',
        'rank_selected_model', 'pairwise_selected_model',
        'risk_selected_model', 'decision_outcome',
    ]
].rename(columns={
    'rank_selected_model': 'baseline_rank_selected_model',
    'pairwise_selected_model': 'baseline_pairwise_selected_model',
    'risk_selected_model': 'baseline_risk_selected_model',
    'decision_outcome': 'baseline_decision_outcome',
})

SENSITIVITY_WITH_BASELINE = SENSITIVITY_FULL_GRID.merge(
    baseline_for_merge,
    on=['dataset', '_dimension_key'],
    how='left',
    validate='many_to_one',
)
SENSITIVITY_WITH_BASELINE['rank_stable'] = (
    SENSITIVITY_WITH_BASELINE['rank_selected_model'].fillna('__ABSTAIN__')
    == SENSITIVITY_WITH_BASELINE['baseline_rank_selected_model'].fillna('__ABSTAIN__')
)
SENSITIVITY_WITH_BASELINE['pairwise_stable'] = (
    SENSITIVITY_WITH_BASELINE['pairwise_selected_model'].fillna('__ABSTAIN__')
    == SENSITIVITY_WITH_BASELINE['baseline_pairwise_selected_model'].fillna('__ABSTAIN__')
)
# Treat two abstentions (both None/NaN) as the same risk outcome.
SENSITIVITY_WITH_BASELINE['risk_stable'] = (
    SENSITIVITY_WITH_BASELINE['risk_selected_model'].fillna('__ABSTAIN__')
    == SENSITIVITY_WITH_BASELINE['baseline_risk_selected_model'].fillna('__ABSTAIN__')
)
SENSITIVITY_WITH_BASELINE['outcome_stable'] = (
    SENSITIVITY_WITH_BASELINE['decision_outcome']
    == SENSITIVITY_WITH_BASELINE['baseline_decision_outcome']
)

SENSITIVITY_STABILITY_BY_DIMENSION = (
    SENSITIVITY_WITH_BASELINE.groupby(['dataset', 'dimension'], as_index=False)
    .agg(
        n_configurations=('config_id', 'nunique'),
        rank_selection_stability=('rank_stable', 'mean'),
        pairwise_selection_stability=('pairwise_stable', 'mean'),
        risk_selection_stability=('risk_stable', 'mean'),
        decision_outcome_stability=('outcome_stable', 'mean'),
        n_distinct_rank_selections=('rank_selected_model', lambda s: s.fillna('__ABSTAIN__').nunique()),
        n_distinct_pairwise_selections=('pairwise_selected_model', lambda s: s.fillna('__ABSTAIN__').nunique()),
        n_distinct_risk_outcomes=('risk_selected_model', lambda s: s.fillna('__ABSTAIN__').nunique()),
    )
)
save_table(SENSITIVITY_STABILITY_BY_DIMENSION, 'reviewer3_sensitivity_stability_by_dimension', latex=True)

SENSITIVITY_STABILITY_OVERALL = pd.DataFrame([{
    'n_dataset_dimensions': int(BASE_DECISIONS.shape[0]),
    'n_configurations': int(SENSITIVITY_FULL_GRID['config_id'].nunique()),
    'rank_selection_stability': float(SENSITIVITY_WITH_BASELINE['rank_stable'].mean()),
    'pairwise_selection_stability': float(SENSITIVITY_WITH_BASELINE['pairwise_stable'].mean()),
    'risk_selection_stability': float(SENSITIVITY_WITH_BASELINE['risk_stable'].mean()),
    'decision_outcome_stability': float(SENSITIVITY_WITH_BASELINE['outcome_stable'].mean()),
}])
save_table(SENSITIVITY_STABILITY_OVERALL, 'reviewer3_sensitivity_stability_overall', latex=True)

# 8b. One-at-a-time sensitivity, easier to describe in the rebuttal.
def one_at_a_time_configs():
    yield ('baseline', 'baseline', BASE_TAU_H, BASE_TAU_L, BASE_FSR_MAX, BASE_MNSQ)
    for v in TAU_H_VALUES:
        yield ('tau_h', f'{v:.2f}', v, BASE_TAU_L, BASE_FSR_MAX, BASE_MNSQ)
    for v in TAU_L_VALUES:
        yield ('tau_l', f'{v:.2f}', BASE_TAU_H, v, BASE_FSR_MAX, BASE_MNSQ)
    for v in FSR_MAX_VALUES:
        yield ('fsr_max', f'{v:.2f}', BASE_TAU_H, BASE_TAU_L, v, BASE_MNSQ)
    for low, high in MNSQ_INTERVALS:
        yield ('mnsq_interval', f'{low:.2f}-{high:.2f}', BASE_TAU_H, BASE_TAU_L, BASE_FSR_MAX, (low, high))


oat_parts = []
for parameter, value, tau_h, tau_l, fsr_max, interval in one_at_a_time_configs():
    d = build_decision_table(SUPPORT_CACHE[(tau_h, tau_l)], fsr_max, interval)
    d['varied_parameter'] = parameter
    d['parameter_value'] = value
    d['tau_h'] = tau_h
    d['tau_l'] = tau_l
    oat_parts.append(d)
ONE_AT_A_TIME = pd.concat(oat_parts, ignore_index=True)
ONE_AT_A_TIME = ONE_AT_A_TIME.merge(
    baseline_for_merge,
    on=['dataset', '_dimension_key'],
    how='left',
    validate='many_to_one',
)
ONE_AT_A_TIME['rank_changed'] = (
    ONE_AT_A_TIME['rank_selected_model'].fillna('__ABSTAIN__')
    != ONE_AT_A_TIME['baseline_rank_selected_model'].fillna('__ABSTAIN__')
)
ONE_AT_A_TIME['pairwise_changed'] = (
    ONE_AT_A_TIME['pairwise_selected_model'].fillna('__ABSTAIN__')
    != ONE_AT_A_TIME['baseline_pairwise_selected_model'].fillna('__ABSTAIN__')
)
ONE_AT_A_TIME['risk_changed'] = (
    ONE_AT_A_TIME['risk_selected_model'].fillna('__ABSTAIN__')
    != ONE_AT_A_TIME['baseline_risk_selected_model'].fillna('__ABSTAIN__')
)
save_table(ONE_AT_A_TIME.drop(columns=['_dimension_key']), 'reviewer3_sensitivity_one_at_a_time')

ONE_AT_A_TIME_SUMMARY = (
    ONE_AT_A_TIME.query("varied_parameter != 'baseline'")
    .groupby(['varied_parameter', 'parameter_value'], as_index=False)
    .agg(
        n_dataset_dimensions=('dimension', 'size'),
        rank_decisions_changed=('rank_changed', 'sum'),
        pairwise_decisions_changed=('pairwise_changed', 'sum'),
        risk_decisions_changed=('risk_changed', 'sum'),
        mean_n_stage2_pass=('n_stage2_pass', 'mean'),
        n_framework_refusals=('framework_refused', 'sum'),
        n_risk_abstentions=('risk_abstained', 'sum'),
    )
)
save_table(ONE_AT_A_TIME_SUMMARY, 'reviewer3_sensitivity_one_at_a_time_summary', latex=True)

# 8c. Stage-2 MNSQ pass sensitivity.
stage2_sensitivity_parts = []
for low, high in MNSQ_INTERVALS:
    gate = stage2_pass_table(low, high)
    stage2_sensitivity_parts.append(
        gate.groupby(['dataset', 'dimension'], as_index=False)
        .agg(
            n_judges=('judge_model', 'nunique'),
            n_stage2_pass=('stage2_pass', 'sum'),
            passing_judges=('judge_model', lambda s: _json_list(
                gate.loc[s.index].loc[gate.loc[s.index, 'stage2_pass'], 'judge_model']
            )),
        )
        .assign(mnsq_low=low, mnsq_high=high)
    )
STAGE2_MNSQ_SENSITIVITY = pd.concat(stage2_sensitivity_parts, ignore_index=True)
save_table(STAGE2_MNSQ_SENSITIVITY, 'reviewer3_stage2_mnsq_sensitivity', latex=True)

# 8d. Metric ranges caused by tau_H/tau_L only.
all_support = pd.concat(SUPPORT_CACHE.values(), ignore_index=True)
SUPPORT_METRIC_RANGES = (
    all_support.groupby(['dataset', 'dimension', 'judge_model'], as_index=False)
    .agg(
        fsr_min=('fsr', 'min'),
        fsr_max=('fsr', 'max'),
        human_sr_min=('human_sr', 'min'),
        human_sr_max=('human_sr', 'max'),
        judge_sr_min=('judge_sr', 'min'),
        judge_sr_max=('judge_sr', 'max'),
        pairacc_tau_min=('pairacc_tau', 'min'),
        pairacc_tau_max=('pairacc_tau', 'max'),
        sda_min=('supported_direction_accuracy', 'min'),
        sda_max=('supported_direction_accuracy', 'max'),
    )
)
save_table(SUPPORT_METRIC_RANGES, 'reviewer3_support_metric_ranges')


# =============================================================================
# 9. DISPLAY THE TABLES MOST USEFUL FOR WRITING THE REBUTTAL
# =============================================================================
print(f'\nSaved all supporting tables to: {OUT}')

print('\nHANNA Stage 1')
display(HANNA_STAGE1)

print('\nHANNA Stage 2 summary')
display(HANNA_STAGE2_SUMMARY)

print('\nHANNA operational decisions')
display(HANNA_DECISIONS)

print('\nReviewer-1 practical decision changes')
display(DECISION_CHANGE_SUMMARY)

print('\nReviewer-3 one-at-a-time sensitivity summary')
display(ONE_AT_A_TIME_SUMMARY)

print('\nReviewer-3 full-grid stability')
display(SENSITIVITY_STABILITY_BY_DIMENSION)

# Dataset-specific summaries make HANNA's contribution visible.
SENSITIVITY_STABILITY_BY_DATASET = (
    SENSITIVITY_WITH_BASELINE.groupby('dataset', as_index=False)
    .agg(n_dataset_dimensions=('dimension', 'nunique'),
         n_configurations=('config_id', 'nunique'),
         rank_selection_stability=('rank_stable', 'mean'),
         pairwise_selection_stability=('pairwise_stable', 'mean'),
         risk_selection_stability=('risk_stable', 'mean'),
         decision_outcome_stability=('outcome_stable', 'mean'))
)
save_table(SENSITIVITY_STABILITY_BY_DATASET, 'reviewer3_sensitivity_stability_by_dataset', latex=True)
display(SENSITIVITY_STABILITY_BY_DATASET)


Loaded datasets: HANNA, HiTZ-BASSE, SummEval
Stage-1 rows: 15 | Stage-2 rows: 135 | Stage-3 rows: 135


/opt/conda/lib/python3.11/site-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.8.1 is installed, but it is not compatible with the installed jaxlib version 0.8.2, so it will not be used.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pairwise rows: 20286

Saved all supporting tables to: /workspaces/TheRulerAndTheJudge-main/notebooks/outputs/rebuttal_supporting_tables

HANNA Stage 1


,dimension,resolution_label,spread80,csem80,delta95,resolution_ratio,theta_reliability,stage1_supported_resolution,stage1_n_supported_pairs,stage1_n_pairs,supported_pairs
0,Coherence,coarse but usable,1.065223,0.112180,0.310948,0.291908,0.918533,0.818182,45,55,45/55
1,Complexity,fine-grained,1.851030,0.111624,0.309407,0.167154,0.926335,0.872727,48,55,48/55
2,Empathy,coarse but usable,0.903142,0.110603,0.306577,0.339456,0.888391,0.854545,47,55,47/55
3,Engagement,fine-grained,1.241697,0.109072,0.302333,0.243484,0.914397,0.872727,48,55,48/55
4,Relevance,coarse but usable,0.928546,0.111277,0.308445,0.332181,0.907893,0.763636,42,55,42/55
5,Surprise,coarse but usable,1.090459,0.125507,0.347888,0.319029,0.892466,0.727273,40,55,40/55



HANNA Stage 2 summary


,dimension,n_judges,n_stage2_pass,passing_judges,min_infit,max_infit,min_outfit,max_outfit
0,Coherence,9,9,"[""deepseek/deepseek-v3.2"", ""meta-llama/llama-3...",0.591120,1.108803,0.702925,1.464065
1,Complexity,9,9,"[""meta-llama/llama-3.3-70b-instruct"", ""qwen/qw...",0.810206,1.115942,0.815529,1.434364
2,Empathy,9,9,"[""deepseek/deepseek-v3.2"", ""meta-llama/llama-3...",0.827608,1.076293,0.851770,1.036371
3,Engagement,9,9,"[""deepseek/deepseek-v3.2"", ""openai/gpt-oss-120...",0.820937,1.196038,0.791896,1.303640
4,Relevance,9,9,"[""deepseek/deepseek-v3.2"", ""meta-llama/llama-3...",0.750232,1.076140,0.763945,1.091199
5,Surprise,9,9,"[""openai/gpt-oss-120b"", ""meta-llama/llama-3.3-...",0.943796,1.124260,0.898315,1.130338


... (5 outputs left out)

,varied_parameter,parameter_value,n_dataset_dimensions,rank_decisions_changed,pairwise_decisions_changed,risk_decisions_changed,mean_n_stage2_pass,n_framework_refusals,n_risk_abstentions
0,fsr_max,0.20,15,0,0,6,6.266667,4,7
1,fsr_max,0.25,15,0,0,4,6.266667,4,6
2,fsr_max,0.30,15,0,0,0,6.266667,4,2
3,fsr_max,0.35,15,0,0,2,6.266667,4,1
4,fsr_max,0.40,15,0,0,2,6.266667,4,1
5,mnsq_interval,0.50-1.30,15,1,1,2,5.400000,4,4
6,mnsq_interval,0.50-1.50,15,0,0,0,6.266667,4,2
7,mnsq_interval,0.50-2.00,15,4,4,0,7.333333,1,5
8,tau_h,0.70,15,0,0,5,6.266667,4,1
9,tau_h,0.75,15,0,0,4,6.266667,4,3



Reviewer-3 full-grid stability


,dataset,dimension,n_configurations,rank_selection_stability,pairwise_selection_stability,risk_selection_stability,decision_outcome_stability,n_distinct_rank_selections,n_distinct_pairwise_selections,n_distinct_risk_outcomes
0,HANNA,Coherence,375,1.000000,1.000000,0.616000,0.896000,1,1,4
1,HANNA,Complexity,375,1.000000,1.000000,0.192000,0.640000,1,1,4
2,HANNA,Empathy,375,1.000000,1.000000,0.560000,1.000000,1,1,2
3,HANNA,Engagement,375,1.000000,1.000000,0.568000,0.600000,1,1,3
4,HANNA,Relevance,375,1.000000,1.000000,0.800000,0.824000,1,1,3
5,HANNA,Surprise,375,1.000000,1.000000,0.952000,0.952000,1,1,2
6,HiTZ-BASSE,5w1h,375,0.666667,0.666667,0.197333,0.272000,2,2,3
7,HiTZ-BASSE,coherence,375,0.666667,0.666667,0.941333,0.666667,2,2,2
8,HiTZ-BASSE,consistency,375,0.666667,0.666667,1.000000,0.666667,2,2,1
9,HiTZ-BASSE,fluency,375,1.000000,1.000000,1.000000,1.000000,1,1,1


,dataset,n_dataset_dimensions,n_configurations,rank_selection_stability,pairwise_selection_stability,risk_selection_stability,decision_outcome_stability
0,HANNA,6,375,1.000000,1.000000,0.614667,0.818667
1,HiTZ-BASSE,5,375,0.733333,0.733333,0.821867,0.654400
2,SummEval,4,375,0.916667,0.916667,0.608000,0.764000


In [3]:
from pathlib import Path

import pandas as pd


ROOT = NOTEBOOK_OUTPUT

OVERALL_PATH = ROOT / "reviewer3_sensitivity_stability_overall.csv"
OAT_PATH = ROOT / "reviewer3_sensitivity_one_at_a_time_summary.csv"

overall = pd.read_csv(OVERALL_PATH)
oat = pd.read_csv(OAT_PATH)


def get_parameter_rows(parameter: str) -> pd.DataFrame:
    """Return one-at-a-time rows in the desired numerical order."""
    rows = oat[oat["varied_parameter"] == parameter].copy()

    if parameter == "mnsq_interval":
        rows["_sort"] = (
            rows["parameter_value"]
            .str.split("-")
            .str[1]
            .astype(float)
        )
    else:
        rows["_sort"] = rows["parameter_value"].astype(float)

    return rows.sort_values("_sort")


def slash_counts(rows: pd.DataFrame, column: str) -> str:
    """Format decision-change counts as '0 / 1 / 2 / ...'."""
    return " / ".join(rows[column].astype(int).astype(str))


def decimal_label(value: str) -> str:
    """Convert 0.70 to .70 for compact rebuttal formatting."""
    return f"{float(value):.2f}".lstrip("0")


def format_tested_values(parameter: str, rows: pd.DataFrame) -> str:
    if parameter == "mnsq_interval":
        labels = []
        for value in rows["parameter_value"]:
            low, high = map(float, value.split("-"))
            low_text = f"{low:.1f}".lstrip("0")
            high_text = f"{high:.1f}".lstrip("0")
            labels.append(f"[{low_text},{high_text}]")
        return " / ".join(labels)

    return " / ".join(decimal_label(v) for v in rows["parameter_value"])


full = overall.iloc[0]

tau_h = get_parameter_rows("tau_h")
tau_l = get_parameter_rows("tau_l")
fsr = get_parameter_rows("fsr_max")
mnsq = get_parameter_rows("mnsq_interval")

compact = pd.DataFrame(
    [
        {
            "Sensitivity analysis": "Full factorial grid",
            "Tested values": f"{int(full['n_configurations'])} configurations",
            "Rank changes": (
                f"{100 * full['rank_selection_stability']:.1f}% stable"
            ),
            "Pairwise changes": (
                f"{100 * full['pairwise_selection_stability']:.1f}% stable"
            ),
            "Risk-selection changes": (
                f"{100 * full['risk_selection_stability']:.1f}% stable"
            ),
        },
        {
            "Sensitivity analysis": r"Human support $\tau_H$",
            "Tested values": format_tested_values("tau_h", tau_h),
            "Rank changes": slash_counts(
                tau_h, "rank_decisions_changed"
            ),
            "Pairwise changes": slash_counts(
                tau_h, "pairwise_decisions_changed"
            ),
            "Risk-selection changes": slash_counts(
                tau_h, "risk_decisions_changed"
            ),
        },
        {
            "Sensitivity analysis": r"Judge support $\tau_L$",
            "Tested values": format_tested_values("tau_l", tau_l),
            "Rank changes": slash_counts(
                tau_l, "rank_decisions_changed"
            ),
            "Pairwise changes": slash_counts(
                tau_l, "pairwise_decisions_changed"
            ),
            "Risk-selection changes": slash_counts(
                tau_l, "risk_decisions_changed"
            ),
        },
        {
            "Sensitivity analysis": r"FSR tolerance $\gamma$",
            "Tested values": format_tested_values("fsr_max", fsr),
            "Rank changes": slash_counts(
                fsr, "rank_decisions_changed"
            ),
            "Pairwise changes": slash_counts(
                fsr, "pairwise_decisions_changed"
            ),
            "Risk-selection changes": slash_counts(
                fsr, "risk_decisions_changed"
            ),
        },
        {
            "Sensitivity analysis": "MNSQ interval",
            "Tested values": format_tested_values(
                "mnsq_interval", mnsq
            ),
            "Rank changes": slash_counts(
                mnsq, "rank_decisions_changed"
            ),
            "Pairwise changes": slash_counts(
                mnsq, "pairwise_decisions_changed"
            ),
            "Risk-selection changes": slash_counts(
                mnsq, "risk_decisions_changed"
            ),
        },
    ]
)

print(compact.to_markdown(index=False))

# Optional outputs
compact.to_csv(ROOT / "reviewer3_compact_sensitivity_table.csv", index=False)

latex = compact.to_latex(
    index=False,
    escape=False,  # preserve \tau and \gamma
    column_format="llrrr",
)
# Preserve math commands while escaping literal percent signs.
latex = latex.replace("%", r"\%")
(ROOT / "reviewer3_compact_sensitivity_table.tex").write_text(
    latex,
    encoding="utf-8",
)

| Sensitivity analysis   | Tested values                  | Rank changes      | Pairwise changes   | Risk-selection changes   |
|:-----------------------|:-------------------------------|:------------------|:-------------------|:-------------------------|
| Full factorial grid    | 375 configurations             | 88.9% stable      | 88.9% stable       | 68.2% stable             |
| Human support $\tau_H$ | .70 / .75 / .80 / .85 / .90    | 0 / 0 / 0 / 0 / 0 | 0 / 0 / 0 / 0 / 0  | 5 / 4 / 0 / 3 / 5        |
| Judge support $\tau_L$ | .70 / .75 / .80 / .85 / .90    | 0 / 0 / 0 / 0 / 0 | 0 / 0 / 0 / 0 / 0  | 3 / 2 / 0 / 1 / 4        |
| FSR tolerance $\gamma$ | .20 / .25 / .30 / .35 / .40    | 0 / 0 / 0 / 0 / 0 | 0 / 0 / 0 / 0 / 0  | 6 / 4 / 0 / 2 / 2        |
| MNSQ interval          | [.5,1.3] / [.5,1.5] / [.5,2.0] | 1 / 0 / 4         | 1 / 0 / 4          | 2 / 0 / 0                |


692

In [4]:
from pathlib import Path
import pandas as pd

RESULTS_DIR = RUNS["HANNA"]
FSR_MAX = 0.30

lb = pd.read_parquet(RESULTS_DIR / "leaderboard_results.parquet").copy()

# Keep the paper-relevant LOOCV rows if these fields are present.
if "Eval split" in lb.columns:
    lb = lb[lb["Eval split"].astype(str).str.casefold() == "loocv"].copy()

if "Outer fold" in lb.columns:
    lb = lb[pd.to_numeric(lb["Outer fold"], errors="coerce").fillna(0) == 0].copy()

# Resolve the actual metric names used by this experiment.
nsmf_col = next(c for c in ["NFSM", "nSMF", "NSMF", "nsmf"] if c in lb.columns)
fsr_col = next(
    c for c in ["False Separation Rate", "Support FSR", "FSR", "fsr"]
    if c in lb.columns
)
raw_rank_col = next(
    c for c in ["Raw-score Spearman", "Raw score Spearman", "raw_score_spearman"]
    if c in lb.columns
)
latent_rank_col = next(c for c in ["Spearman", "spearman"] if c in lb.columns)

rows = []

for dimension in ["Complexity", "Relevance"]:
    g = lb[
        lb["Dimension"].astype(str).str.casefold() == dimension.casefold()
    ].copy()

    risk_eligible = g[g[fsr_col] <= FSR_MAX]

    winners = {
        "Raw correlation": g.loc[g[raw_rank_col].idxmax()],
        "Calibrated rank": g.loc[g[latent_rank_col].idxmax()],
        "Pairwise fidelity": g.loc[g[nsmf_col].idxmax()],
        "Risk-sensitive": risk_eligible.loc[risk_eligible[nsmf_col].idxmax()],
    }

    for objective, winner in winners.items():
        rows.append({
            "dimension": dimension,
            "objective": objective,
            "judge": winner["Model"],
            "raw_spearman": winner[raw_rank_col],
            "latent_spearman": winner[latent_rank_col],
            "nsmf": winner[nsmf_col],
            "fsr": winner[fsr_col],
        })

summary = pd.DataFrame(rows)

display(
    summary.style.format({
        "raw_spearman": "{:.3f}",
        "latent_spearman": "{:.3f}",
        "nsmf": "{:.3f}",
        "fsr": "{:.3f}",
    })
)

print(summary.to_markdown(index=False, floatfmt=".3f"))

,dimension,objective,judge,raw_spearman,latent_spearman,nsmf,fsr
0,Complexity,Raw correlation,deepseek/deepseek-v3.2,0.945,0.873,0.801,0.571
1,Complexity,Calibrated rank,openai/gpt-oss-20b,0.943,0.936,0.855,0.429
2,Complexity,Pairwise fidelity,openai/gpt-oss-20b,0.943,0.936,0.855,0.429
3,Complexity,Risk-sensitive,google/gemma-3-27b-it,0.804,0.827,0.678,0.286
4,Relevance,Raw correlation,qwen/qwen3-14b,0.729,0.782,0.862,0.500
5,Relevance,Calibrated rank,qwen/qwen3-14b,0.729,0.782,0.862,0.500
6,Relevance,Pairwise fidelity,qwen/qwen3-14b,0.729,0.782,0.862,0.500
7,Relevance,Risk-sensitive,openai/gpt-oss-20b,0.653,0.691,0.821,0.214


| dimension   | objective         | judge                  |   raw_spearman |   latent_spearman |   nsmf |   fsr |
|:------------|:------------------|:-----------------------|---------------:|------------------:|-------:|------:|
| Complexity  | Raw correlation   | deepseek/deepseek-v3.2 |          0.945 |             0.873 |  0.801 | 0.571 |
| Complexity  | Calibrated rank   | openai/gpt-oss-20b     |          0.943 |             0.936 |  0.855 | 0.429 |
| Complexity  | Pairwise fidelity | openai/gpt-oss-20b     |          0.943 |             0.936 |  0.855 | 0.429 |
| Complexity  | Risk-sensitive    | google/gemma-3-27b-it  |          0.804 |             0.827 |  0.678 | 0.286 |
| Relevance   | Raw correlation   | qwen/qwen3-14b         |          0.729 |             0.782 |  0.862 | 0.500 |
| Relevance   | Calibrated rank   | qwen/qwen3-14b         |          0.729 |             0.782 |  0.862 | 0.500 |
| Relevance   | Pairwise fidelity | qwen/qwen3-14b         |          0.729 |   

In [5]:
# Strict versus exact-coordinate mid-dominance on all 15 settings.
# Historical 1/9 conclusions are not imposed on the expanded result set.
from pathlib import Path

import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------

COMPARISON_DATASETS = {
    "SummEval": {
        "coherence",
        "consistency",
        "fluency",
        "relevance",
    },
    "HiTZ-BASSE": {
        "5w1h",
        "coherence",
        "consistency",
        "fluency",
        "relevance",
    },
    "HANNA": {"coherence", "complexity", "empathy", "engagement", "relevance", "surprise"},
}

OUT = Path(OUT)
OUT.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def require_columns(
    frame: pd.DataFrame,
    columns: list[str],
    frame_name: str,
) -> None:
    missing = set(columns) - set(frame.columns)

    if missing:
        raise KeyError(
            f"{frame_name} is missing columns {sorted(missing)}. "
            f"Available columns: {list(frame.columns)}"
        )


def format_model_set(models: frozenset[str]) -> str:
    return " | ".join(sorted(models))


def risk_state(
    framework_refused: bool,
    risk_abstained: bool,
    selected_model,
) -> str:
    if bool(framework_refused):
        return "REFUSE"

    if bool(risk_abstained) or pd.isna(selected_model):
        return "ABSTAIN"

    return f"SELECT: {selected_model}"


def select_minimum_fsr_sets(
    support: pd.DataFrame,
    *,
    atol: float = 1e-12,
) -> pd.DataFrame:
    """
    Return the complete set of FSR-minimizing judges for every
    dataset–dimension setting.

    This deliberately retains ties. It does not choose an arbitrary
    alphabetical representative from a tied minimum.
    """
    require_columns(
        support,
        [
            "dataset",
            "_dimension_key",
            "fsr",
        ],
        "support",
    )

    if "judge_model" in support.columns:
        model_column = "judge_model"
    elif "_model_key" in support.columns:
        model_column = "_model_key"
    else:
        raise KeyError(
            "Support table contains neither 'judge_model' nor '_model_key'."
        )

    rows = []

    for (dataset, dimension_key), group in support.groupby(
        ["dataset", "_dimension_key"],
        sort=True,
    ):
        valid = group.dropna(subset=["fsr"]).copy()

        if valid.empty:
            rows.append(
                {
                    "dataset": dataset,
                    "_dimension_key": dimension_key,
                    "minimum_fsr": np.nan,
                    "fsr_minimizer_set": frozenset(),
                    "n_fsr_minimizers": 0,
                }
            )
            continue

        minimum_fsr = float(valid["fsr"].min())

        minimizer_rows = valid[
            np.isclose(
                valid["fsr"],
                minimum_fsr,
                rtol=0.0,
                atol=atol,
            )
        ]

        minimizer_set = frozenset(
            minimizer_rows[model_column].astype(str)
        )

        rows.append(
            {
                "dataset": dataset,
                "_dimension_key": dimension_key,
                "minimum_fsr": minimum_fsr,
                "fsr_minimizer_set": minimizer_set,
                "n_fsr_minimizers": len(minimizer_set),
            }
        )

    return pd.DataFrame(rows)


# -----------------------------------------------------------------------------
# 1. Restrict the comparison to the all configured settings
# -----------------------------------------------------------------------------

required_pairwise_columns = [
    "dataset",
    "_dimension_key",
    "_model_key",
    "system_a",
    "system_b",
    "human_p_gt_strict",
    "human_p_equal_grid",
    "judge_p_gt_strict",
    "judge_p_equal_grid",
    "human_signed_margin",
    "judge_signed_margin",
]

require_columns(
    PAIRWISE,
    required_pairwise_columns,
    "PAIRWISE",
)

pairwise_comparison = PAIRWISE[
    PAIRWISE["dataset"].isin(COMPARISON_DATASETS)
].copy()

observed_settings = (
    pairwise_comparison[
        ["dataset", "_dimension_key"]
    ]
    .drop_duplicates()
    .sort_values(["dataset", "_dimension_key"])
)

observed_dimensions = {
    dataset: set(
        observed_settings.loc[
            observed_settings["dataset"] == dataset,
            "_dimension_key",
        ]
    )
    for dataset in COMPARISON_DATASETS
}

assert observed_dimensions == COMPARISON_DATASETS, (
    "The posterior artifacts do not contain the expected configured settings.\n"
    f"Expected: {COMPARISON_DATASETS}\n"
    f"Observed: {observed_dimensions}"
)

assert len(observed_settings) == sum(map(len, COMPARISON_DATASETS.values())), (
    f"Expected all configured settings, found {len(observed_settings)}."
)


# -----------------------------------------------------------------------------
# 2. Verify the existing PAIRWISE margins use mid-dominance
# -----------------------------------------------------------------------------

expected_human_mid_margin = (
    2.0
    * (
        pairwise_comparison["human_p_gt_strict"]
        + 0.5 * pairwise_comparison["human_p_equal_grid"]
    )
    - 1.0
)

expected_judge_mid_margin = (
    2.0
    * (
        pairwise_comparison["judge_p_gt_strict"]
        + 0.5 * pairwise_comparison["judge_p_equal_grid"]
    )
    - 1.0
)

assert np.allclose(
    pairwise_comparison["human_signed_margin"],
    expected_human_mid_margin,
    rtol=0.0,
    atol=1e-12,
), "PAIRWISE human margins do not match mid-dominance."

assert np.allclose(
    pairwise_comparison["judge_signed_margin"],
    expected_judge_mid_margin,
    rtol=0.0,
    atol=1e-12,
), "PAIRWISE judge margins do not match mid-dominance."


# -----------------------------------------------------------------------------
# 3. Reconstruct the strict convention from the same probabilities
# -----------------------------------------------------------------------------

pairwise_strict = pairwise_comparison.copy()

pairwise_strict["human_signed_margin"] = (
    2.0 * pairwise_strict["human_p_gt_strict"] - 1.0
)

pairwise_strict["judge_signed_margin"] = (
    2.0 * pairwise_strict["judge_p_gt_strict"] - 1.0
)


# Confirm that the paired comparison changes only the margin convention.
pair_key = [
    "dataset",
    "_dimension_key",
    "_model_key",
    "system_a",
    "system_b",
]

assert (
    pairwise_comparison[pair_key]
    .reset_index(drop=True)
    .equals(
        pairwise_strict[pair_key]
        .reset_index(drop=True)
    )
)

for column in [
    "human_p_gt_strict",
    "human_p_equal_grid",
    "judge_p_gt_strict",
    "judge_p_equal_grid",
]:
    assert np.array_equal(
        pairwise_comparison[column].to_numpy(),
        pairwise_strict[column].to_numpy(),
        equal_nan=True,
    ), f"{column} changed between the paired calculations."


# -----------------------------------------------------------------------------
# 4. Recompute support quantities under both conventions
# -----------------------------------------------------------------------------

strict_support = support_metrics_for_thresholds(
    pairwise_strict,
    tau_h=BASE_TAU_H,
    tau_l=BASE_TAU_L,
)

mid_support = support_metrics_for_thresholds(
    pairwise_comparison,
    tau_h=BASE_TAU_H,
    tau_l=BASE_TAU_L,
)


# Keep only the configured datasets defensively, in case the helper carries
# additional rows from shared/global tables.
strict_support = strict_support[
    strict_support["dataset"].isin(COMPARISON_DATASETS)
].copy()

mid_support = mid_support[
    mid_support["dataset"].isin(COMPARISON_DATASETS)
].copy()


support_key = [
    "dataset",
    "_dimension_key",
    "_model_key",
]

assert (
    strict_support[support_key]
    .sort_values(support_key)
    .reset_index(drop=True)
    .equals(
        mid_support[support_key]
        .sort_values(support_key)
        .reset_index(drop=True)
    )
), "Strict and mid support tables do not contain identical judges/settings."


# -----------------------------------------------------------------------------
# 5. Compare the complete sets of FSR minimizers
# -----------------------------------------------------------------------------

strict_minima = (
    select_minimum_fsr_sets(strict_support)
    .rename(
        columns={
            "minimum_fsr": "strict_minimum_fsr",
            "fsr_minimizer_set": "strict_fsr_minimizer_set",
            "n_fsr_minimizers": "strict_n_fsr_minimizers",
        }
    )
)

mid_minima = (
    select_minimum_fsr_sets(mid_support)
    .rename(
        columns={
            "minimum_fsr": "mid_minimum_fsr",
            "fsr_minimizer_set": "mid_fsr_minimizer_set",
            "n_fsr_minimizers": "mid_n_fsr_minimizers",
        }
    )
)

minimum_comparison = strict_minima.merge(
    mid_minima,
    on=["dataset", "_dimension_key"],
    how="inner",
    validate="one_to_one",
)


# The relevant invariance claim:
# every original minimizer remains a minimizer after the correction.
minimum_comparison["all_original_minimizers_retained"] = (
    minimum_comparison.apply(
        lambda row: (
            row["strict_fsr_minimizer_set"]
            <= row["mid_fsr_minimizer_set"]
        ),
        axis=1,
    )
)

# Stronger property, reported separately:
# whether the complete tied argmin set is exactly identical.
minimum_comparison["minimizer_set_exactly_unchanged"] = (
    minimum_comparison["strict_fsr_minimizer_set"]
    == minimum_comparison["mid_fsr_minimizer_set"]
)

minimum_comparison["strict_fsr_minimizers"] = (
    minimum_comparison[
        "strict_fsr_minimizer_set"
    ].map(format_model_set)
)

minimum_comparison["mid_fsr_minimizers"] = (
    minimum_comparison[
        "mid_fsr_minimizer_set"
    ].map(format_model_set)
)


# -----------------------------------------------------------------------------
# 6. Recompute the operational risk-sensitive outcomes
# -----------------------------------------------------------------------------

strict_decisions = build_decision_table(
    strict_support,
    fsr_max=BASE_FSR_MAX,
    mnsq_interval=BASE_MNSQ,
)

mid_decisions = build_decision_table(
    mid_support,
    fsr_max=BASE_FSR_MAX,
    mnsq_interval=BASE_MNSQ,
)


# build_decision_table may start from shared global experiment tables, so
# explicitly retain all configured datasets, including HANNA.
strict_decisions = strict_decisions[
    strict_decisions["dataset"].isin(COMPARISON_DATASETS)
].copy()

mid_decisions = mid_decisions[
    mid_decisions["dataset"].isin(COMPARISON_DATASETS)
].copy()


decision_columns = [
    "dataset",
    "_dimension_key",
    "dimension",
    "framework_refused",
    "risk_abstained",
    "risk_selected_model",
    "risk_selected_nsmf",
    "risk_selected_fsr",
    "decision_outcome",
]

require_columns(
    strict_decisions,
    decision_columns,
    "strict_decisions",
)

require_columns(
    mid_decisions,
    decision_columns,
    "mid_decisions",
)


strict_decisions = strict_decisions[
    decision_columns
].rename(
    columns={
        "dimension": "strict_dimension",
        "framework_refused": "strict_framework_refused",
        "risk_abstained": "strict_risk_abstained",
        "risk_selected_model": "strict_risk_selected_model",
        "risk_selected_nsmf": "strict_risk_selected_nsmf",
        "risk_selected_fsr": "strict_risk_selected_fsr",
        "decision_outcome": "strict_decision_outcome",
    }
)

mid_decisions = mid_decisions[
    decision_columns
].rename(
    columns={
        "dimension": "mid_dimension",
        "framework_refused": "mid_framework_refused",
        "risk_abstained": "mid_risk_abstained",
        "risk_selected_model": "mid_risk_selected_model",
        "risk_selected_nsmf": "mid_risk_selected_nsmf",
        "risk_selected_fsr": "mid_risk_selected_fsr",
        "decision_outcome": "mid_decision_outcome",
    }
)

decision_comparison = strict_decisions.merge(
    mid_decisions,
    on=["dataset", "_dimension_key"],
    how="inner",
    validate="one_to_one",
)

decision_comparison["dimension"] = (
    decision_comparison["mid_dimension"]
)

decision_comparison["strict_risk_state"] = (
    decision_comparison.apply(
        lambda row: risk_state(
            row["strict_framework_refused"],
            row["strict_risk_abstained"],
            row["strict_risk_selected_model"],
        ),
        axis=1,
    )
)

decision_comparison["mid_risk_state"] = (
    decision_comparison.apply(
        lambda row: risk_state(
            row["mid_framework_refused"],
            row["mid_risk_abstained"],
            row["mid_risk_selected_model"],
        ),
        axis=1,
    )
)

decision_comparison["risk_outcome_changed"] = (
    decision_comparison["strict_risk_state"]
    != decision_comparison["mid_risk_state"]
)

decision_comparison["selection_to_abstention"] = (
    decision_comparison[
        "strict_risk_state"
    ].str.startswith("SELECT:")
    & decision_comparison[
        "mid_risk_state"
    ].eq("ABSTAIN")
)


# -----------------------------------------------------------------------------
# 7. Construct one auditable claim table
# -----------------------------------------------------------------------------

claim_table = minimum_comparison.merge(
    decision_comparison[
        [
            "dataset",
            "_dimension_key",
            "dimension",
            "strict_risk_state",
            "strict_risk_selected_model",
            "strict_risk_selected_fsr",
            "mid_risk_state",
            "mid_risk_selected_model",
            "mid_risk_selected_fsr",
            "risk_outcome_changed",
            "selection_to_abstention",
        ]
    ],
    on=["dataset", "_dimension_key"],
    how="inner",
    validate="one_to_one",
)

claim_table = claim_table.sort_values(
    ["dataset", "_dimension_key"]
).reset_index(drop=True)


# -----------------------------------------------------------------------------
# 8. Validate coverage; outcome counts below are data, not assumptions.
# -----------------------------------------------------------------------------
assert len(claim_table) == sum(map(len, COMPARISON_DATASETS.values()))
assert claim_table.groupby('dataset').size().to_dict() == {
    dataset: len(dimensions) for dataset, dimensions in COMPARISON_DATASETS.items()
}
changed = claim_table[claim_table['risk_outcome_changed']].copy()

# -----------------------------------------------------------------------------
# 10. Produce compact summary
# -----------------------------------------------------------------------------

claim_summary = pd.DataFrame(
    [
        {
            "n_dataset_dimensions": len(claim_table),
            "settings_where_original_minimizers_retained": int(
                claim_table[
                    "all_original_minimizers_retained"
                ].sum()
            ),
            "settings_with_exactly_same_argmin_set": int(
                claim_table[
                    "minimizer_set_exactly_unchanged"
                ].sum()
            ),
            "risk_outcomes_unchanged": int(
                (~claim_table["risk_outcome_changed"]).sum()
            ),
            "risk_outcomes_changed": int(
                claim_table["risk_outcome_changed"].sum()
            ),
            "selection_to_abstention": int(
                claim_table["selection_to_abstention"].sum()
            ),
        }
    ]
)


display_columns = [
    "dataset",
    "_dimension_key",
    "strict_fsr_minimizers",
    "strict_minimum_fsr",
    "mid_fsr_minimizers",
    "mid_minimum_fsr",
    "all_original_minimizers_retained",
    "strict_risk_state",
    "mid_risk_state",
    "risk_outcome_changed",
]

print("\nStrict versus mid-dominance comparison:")
print(
    claim_table[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nClaim summary:")
print(claim_summary.to_string(index=False))

print("\nChanged setting:")
print(
    changed[
        display_columns
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)


# -----------------------------------------------------------------------------
# 11. Save auditable outputs
# -----------------------------------------------------------------------------

# Remove Python set objects from the CSV-facing copy.
claim_table_csv = claim_table.drop(
    columns=[
        "strict_fsr_minimizer_set",
        "mid_fsr_minimizer_set",
    ]
)

claim_table_csv.to_csv(
    OUT / "reviewer2_strict_vs_mid_all_datasets.csv",
    index=False,
)

claim_summary.to_csv(
    OUT / "reviewer2_strict_vs_mid_all_datasets_summary.csv",
    index=False,
)

print(
    "\nSaved:",
    OUT / "reviewer2_strict_vs_mid_all_datasets.csv",
)

print(
    "Saved:",
    OUT / "reviewer2_strict_vs_mid_all_datasets_summary.csv",
)
# Same claims summarized separately for each dataset.
by_dataset = claim_table.groupby('dataset', as_index=False).agg(
    n_dataset_dimensions=('dimension', 'size'),
    settings_where_original_minimizers_retained=('all_original_minimizers_retained', 'sum'),
    settings_with_exactly_same_argmin_set=('minimizer_set_exactly_unchanged', 'sum'),
    risk_outcomes_changed=('risk_outcome_changed', 'sum'),
    selection_to_abstention=('selection_to_abstention', 'sum'),
)
save_table(by_dataset, 'reviewer2_strict_vs_mid_by_dataset', latex=True)
display(by_dataset)



Strict versus mid-dominance comparison:
   dataset _dimension_key                                     strict_fsr_minimizers  strict_minimum_fsr                                      mid_fsr_minimizers  mid_minimum_fsr  all_original_minimizers_retained              strict_risk_state                 mid_risk_state  risk_outcome_changed
     HANNA      coherence deepseek/deepseek-v3.2 | meta-llama/llama-3.2-3b-instruct            0.200000                        meta-llama/llama-3.2-3b-instruct         0.000000                             False SELECT: deepseek/deepseek-v3.2 SELECT: deepseek/deepseek-v3.2                 False
     HANNA     complexity                                     google/gemma-3-27b-it            0.166667                                   google/gemma-3-27b-it         0.285714                              True  SELECT: google/gemma-3-27b-it  SELECT: google/gemma-3-27b-it                 False
     HANNA        empathy                                      google/gemm

,dataset,n_dataset_dimensions,settings_where_original_minimizers_retained,settings_with_exactly_same_argmin_set,risk_outcomes_changed,selection_to_abstention
0,HANNA,6,5,5,2,0
1,HiTZ-BASSE,5,5,5,0,0
2,SummEval,4,4,3,1,1


### Publication LaTeX for rebuttal sensitivity analyses


In [7]:
from notebooks.export_sensitivity_latex import write_sensitivity_latex

for path in write_sensitivity_latex(NOTEBOOK_OUTPUT):
    print(f"Wrote {path}")


Wrote /workspaces/TheRulerAndTheJudge-main/notebooks/outputs/rebuttal_supporting_tables/rebuttal_sensitivity_summary.tex
Wrote /workspaces/TheRulerAndTheJudge-main/notebooks/outputs/rebuttal_supporting_tables/rebuttal_sensitivity_tables.tex
